# ops

> Queue-touching layer operations: the shared `graph_task` helper (task channel), idempotent emission (emit-if-absent + verify-if-present), and `extend_graph` — the one primitive every graph-extending workflow commits through. Deterministic ids (see `identity`) make idempotency a presence check instead of a search.

In [ ]:
#| default_exp ops

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Set, Tuple

from cjm_plugin_system.core.queue import JobQueue, JobStatus
# Importing the typed query/result classes IS the host-side wire registration (F8).
from cjm_context_graph_primitives.query import NodeQuery, EdgeQuery, NodeQueryResult, EdgeQueryResult

In [ ]:
#| export
GRAPH_TASK = "graph-storage"  # The graph-storage adapter task (explicit task channel, stage 4)


async def graph_task(
    queue: JobQueue,  # Started job queue
    graph_id: str,    # Graph-storage capability instance id
    method: str,      # Adapter method (e.g. "query_nodes", "add_nodes")
    **kwargs,         # Typed-method kwargs (wire dicts ok; the in-worker adapter normalizes)
) -> Any:  # Typed task result (wire-decoded host-side)
    """Invoke a graph-storage adapter method through the queue's task channel.

    THE shared copy: decomp-core and correction-core's per-core helpers migrate
    onto this one (graph ops stay on the queue path for telemetry/cancellation
    per D7/Thread-5 lock 5).
    """
    jid = await queue.submit(graph_id, task=GRAPH_TASK, method=method, **kwargs)
    job = await queue.wait_for_job(jid)
    if job.status != JobStatus.completed:
        raise RuntimeError(f"{graph_id} {method} {job.status}: {job.error}")
    return job.result

In [ ]:
#| export
class GraphIntegrityError(RuntimeError):
    """An emitted node collided with an existing node of different identity content.

    Raised by verify-if-present: same deterministic id but mismatched label or
    provenance content hashes means the identity tuple and the content have
    diverged — never overwrite silently."""
    pass

In [ ]:
#| export
def _source_hashes(sources: Optional[List[Any]]) -> Set[str]:
    """Content-hash set from a node's sources (typed SourceRefs or wire dicts)."""
    out: Set[str] = set()
    for s in sources or []:
        h = s.get("content_hash") if isinstance(s, dict) else getattr(s, "content_hash", None)
        if h:
            out.add(h)
    return out


def node_identity_mismatch(
    existing: Any,            # Existing node (typed GraphNode or wire dict)
    new: Dict[str, Any],      # New node wire dict being emitted
) -> Optional[str]:  # Mismatch description, or None when compatible
    """Verify-if-present check: label + sources content-hash set must match."""
    ex_label = existing.get("label") if isinstance(existing, dict) else getattr(existing, "label", None)
    if ex_label != new.get("label"):
        return f"label mismatch: existing {ex_label!r} != new {new.get('label')!r}"
    ex_sources = existing.get("sources") if isinstance(existing, dict) else getattr(existing, "sources", None)
    ex_hashes, new_hashes = _source_hashes(ex_sources), _source_hashes(new.get("sources"))
    if ex_hashes != new_hashes:
        return f"sources content-hash mismatch: existing {sorted(ex_hashes)} != new {sorted(new_hashes)}"
    return None

In [ ]:
#| export
def partition_by_presence(
    items: List[Dict[str, Any]],  # Wire dicts carrying "id"
    existing_ids: Set[str],       # Ids already present in the graph
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:  # (absent, present)
    """Split wire dicts into absent (to add) and present (to verify)."""
    absent = [it for it in items if it["id"] not in existing_ids]
    present = [it for it in items if it["id"] in existing_ids]
    return absent, present

In [ ]:
#| export
@dataclass
class ExtendResult:
    """Outcome of one idempotent extend_graph commit."""
    nodes_added: int = 0      # Nodes newly created
    nodes_verified: int = 0   # Nodes already present, identity-verified
    edges_added: int = 0      # Edges newly created
    edges_existing: int = 0   # Edges already present (skipped)
    added_node_ids: List[str] = field(default_factory=list)  # Ids of created nodes
    added_edge_ids: List[str] = field(default_factory=list)  # Ids of created edges

In [ ]:
#| export
async def extend_graph(
    queue: JobQueue,              # Started job queue
    graph_id: str,                # Graph-storage capability id
    nodes: List[Dict[str, Any]],  # Node wire dicts (deterministic ids for layer-0; generated for decisions)
    edges: List[Dict[str, Any]],  # Edge wire dicts
) -> ExtendResult:  # Counts + created ids
    """Idempotently extend the graph: emit-if-absent + verify-if-present.

    Deterministic ids make idempotency a batched presence check (2 reads + at
    most 2 writes per call — the C17 lesson applied to the write path): nodes
    already present are verified against the new emission (label + provenance
    content hashes) and a mismatch raises `GraphIntegrityError` LOUDLY; absent
    nodes/edges are added. Cache-hit re-emission therefore collides into a
    verified no-op (stress item 4), and a re-derived spine reproduces — never
    duplicates — its layer-0 (stress item 1).
    """
    result = ExtendResult()

    if nodes:
        res = await graph_task(queue, graph_id, "query_nodes",
                               query=NodeQuery(ids=[n["id"] for n in nodes]).to_dict())
        existing = {gn.id: gn for gn in (res.nodes or [])}
        absent, present = partition_by_presence(nodes, set(existing))
        for n in present:
            msg = node_identity_mismatch(existing[n["id"]], n)
            if msg:
                raise GraphIntegrityError(f"node {n['id']}: {msg}")
        result.nodes_verified = len(present)
        if absent:
            added = await graph_task(queue, graph_id, "add_nodes", nodes=absent)
            result.added_node_ids = list(added or [])
            result.nodes_added = len(result.added_node_ids)

    if edges:
        eres = await graph_task(queue, graph_id, "query_edges",
                                query=EdgeQuery(ids=[e["id"] for e in edges], project=["id"]).to_dict())
        existing_eids = {r["id"] for r in (eres.rows or [])}
        absent_edges = [e for e in edges if e["id"] not in existing_eids]
        result.edges_existing = len(edges) - len(absent_edges)
        if absent_edges:
            added = await graph_task(queue, graph_id, "add_edges", edges=absent_edges)
            result.added_edge_ids = list(added or [])
            result.edges_added = len(result.added_edge_ids)

    return result

In [ ]:
# tests — pure helpers (the async path is exercised live by the cores' loop-backs)
absent, present = partition_by_presence([{"id": "a"}, {"id": "b"}], {"b"})
assert [n["id"] for n in absent] == ["a"] and [n["id"] for n in present] == ["b"]

existing = {"label": "Source", "sources": [{"content_hash": "sha256:x"}]}
new_ok = {"label": "Source", "sources": [{"content_hash": "sha256:x"}]}
new_label = {"label": "Doc", "sources": [{"content_hash": "sha256:x"}]}
new_hash = {"label": "Source", "sources": [{"content_hash": "sha256:y"}]}
assert node_identity_mismatch(existing, new_ok) is None
assert "label mismatch" in node_identity_mismatch(existing, new_label)
assert "content-hash mismatch" in node_identity_mismatch(existing, new_hash)

# typed-object tolerance (GraphNode-shaped)
class _FakeNode:
    label = "Source"
    sources = [type("S", (), {"content_hash": "sha256:x"})()]
assert node_identity_mismatch(_FakeNode(), new_ok) is None

r = ExtendResult()
assert r.nodes_added == 0 and r.added_edge_ids == []
print("ops tests OK")